# Ablation Study 2 — Feature Mode: Topology vs Full

**Question:** Do the extra 6 node features (aromatic flag, hybridization, H count, formal charge) materially improve U0 prediction over atom-type one-hots alone?  
**Models:** GCN, GAT, GATv2  
**Variable:** `feature_mode` ∈ {'topology', 'full'}  
- `topology`: node = [N, 5] atom-type one-hot only, edge = [E, 4] bond type  
- `full`:     node = [N, 11] all features, edge = [E, 4] bond type  
**Fixed:** All other hyperparameters constant.  

**Output directory:** `MyDrive/Ablation/Study2_FeatureMode/`

In [ ]:
# ── Cell 1: Mount Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 2: Install dependencies ───────────────────────────────────────────────
import torch, subprocess, sys
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')
torch_version = torch.__version__.split('+')[0]
cuda_tag = 'cu121' if torch.cuda.is_available() else 'cpu'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-scatter', 'torch-sparse',
                '-f', f'https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric'], check=True)
print('PyG installed.')

In [ ]:
# ── Cell 3: Clone repo & set paths ────────────────────────────────────────────
import os, sys, subprocess

REPO_URL  = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'  # ← update this
REPO_DIR  = '/content/gnn_project'
DRIVE_OUT = '/content/drive/MyDrive/Ablation/Study2_FeatureMode'
DATA_ROOT = '/content/qm9_data'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

for sub in ['checkpoints', 'logs', 'results', 'plots']:
    os.makedirs(f'{DRIVE_OUT}/{sub}', exist_ok=True)

print(f'Output root: {DRIVE_OUT}')

In [ ]:
# ── Cell 4: USER CONFIG ───────────────────────────────────────────────────────
# Edit these values directly.

# ---------- Training ----------
EPOCHS      = 100
PATIENCE    = 15
BATCH_SIZE  = 128
LR          = 5e-4

# ---------- Model (fixed — use best known config) ----------
HIDDEN_DIM  = 256
NUM_LAYERS  = 6      # use best from Study 1, or fix at 6
DROPOUT     = 0.0
HEADS       = 8

# ---------- Ablation variable ----------
FEATURE_MODES = ['topology', 'full']
MODELS        = ['gcn', 'gat', 'gatv2']

# ---------- Dataset ----------
TARGET_IDX  = 7
SEED        = 42
SPLIT       = [0.8, 0.1, 0.1]

print('Config set.')

In [ ]:
# ── Cell 5: Training functions ────────────────────────────────────────────────
import copy, csv, torch
import torch.nn.functional as F
from tqdm import tqdm
from data.features import select_features, get_feature_dims
from data.loader import get_dataloaders
from models import build_model

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

def mae(pred, target):
    return (pred - target).abs().mean().item()


def run_epoch(model, loader, optimizer, device, normalizer, feature_mode, model_name, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_targets = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            batch = select_features(batch, mode=feature_mode)
            batch = batch.to(device)
            if model_name == 'gatv2':
                pred = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
            else:
                pred = model(batch.x, batch.edge_index, batch.batch)
            target = batch.y.view(-1)
            loss = F.mse_loss(pred, target)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * batch.num_graphs
            all_preds.append(normalizer.denormalize(pred.detach().cpu()))
            all_targets.append(normalizer.denormalize(target.detach().cpu()))
    n = sum(t.size(0) for t in all_targets)
    return total_loss / n, mae(torch.cat(all_preds), torch.cat(all_targets))


def train_run(model_name, feature_mode, run_id, train_loader, val_loader, normalizer):
    feature_dims = get_feature_dims(feature_mode)
    cfg = {
        'dataset':  {'target': TARGET_IDX, 'split': SPLIT, 'seed': SEED, 'feature_mode': feature_mode},
        'training': {'batch_size': BATCH_SIZE, 'epochs': EPOCHS, 'patience': PATIENCE, 'lr': LR},
        'model':    {'hidden_dim': HIDDEN_DIM, 'num_layers': NUM_LAYERS, 'dropout': DROPOUT, 'heads': HEADS}
    }
    model = build_model(model_name, cfg, feature_dims=feature_dims).to(DEVICE)
    param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  [{run_id}] {model_name} | mode={feature_mode} | params={param_count:,}')

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    ckpt_path = f'{DRIVE_OUT}/checkpoints/{run_id}_best.pt'
    log_path  = f'{DRIVE_OUT}/logs/{run_id}_history.csv'

    best_val_mae, best_state, patience_ctr = float('inf'), None, 0
    history = []

    with open(log_path, 'w', newline='') as f:
        csv.DictWriter(f, fieldnames=['epoch','train_loss','val_loss','val_mae','lr']).writeheader()

    pbar = tqdm(range(1, EPOCHS + 1), desc=run_id, leave=True)
    for epoch in pbar:
        tr_loss, _ = run_epoch(model, train_loader, optimizer, DEVICE, normalizer, feature_mode, model_name, train=True)
        vl_loss, vl_mae_val = run_epoch(model, val_loader, None, DEVICE, normalizer, feature_mode, model_name, train=False)
        lr_now = optimizer.param_groups[0]['lr']
        row = {'epoch': epoch, 'train_loss': f'{tr_loss:.6f}', 'val_loss': f'{vl_loss:.6f}',
               'val_mae': f'{vl_mae_val:.6f}', 'lr': f'{lr_now:.2e}'}
        history.append(row)
        with open(log_path, 'a', newline='') as f:
            csv.DictWriter(f, fieldnames=['epoch','train_loss','val_loss','val_mae','lr']).writerow(row)
        pbar.set_postfix(val_mae=f'{vl_mae_val:.4f}')
        if vl_mae_val < best_val_mae:
            best_val_mae = vl_mae_val
            best_state   = copy.deepcopy(model.state_dict())
            patience_ctr = 0
            torch.save(best_state, ckpt_path)
        else:
            patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'  Early stop at epoch {epoch}')
            break

    print(f'  [{run_id}] best val MAE = {best_val_mae:.4f} Ha')
    return {'run_id': run_id, 'model': model_name, 'feature_mode': feature_mode,
            'best_val_mae': best_val_mae, 'params': param_count, 'checkpoint': ckpt_path}


print('Functions defined.')

In [ ]:
# ── Cell 6: Run all combinations ─────────────────────────────────────────────
# NOTE: Data is reloaded per feature_mode because node_dim changes (5 vs 11).
# The split seed is fixed so splits are identical across both modes.
import pandas as pd

results = []
results_path = f'{DRIVE_OUT}/results/study2_results.csv'
total = len(MODELS) * len(FEATURE_MODES)
done  = 0

for feature_mode in FEATURE_MODES:
    # Reload data for this feature mode (node_dim differs)
    base_cfg = {
        'dataset':  {'target': TARGET_IDX, 'split': SPLIT, 'seed': SEED, 'feature_mode': feature_mode},
        'training': {'batch_size': BATCH_SIZE, 'epochs': EPOCHS, 'patience': PATIENCE, 'lr': LR},
    }
    print(f'\nLoading data for feature_mode={feature_mode} ...')
    train_loader, val_loader, _, normalizer = get_dataloaders(base_cfg, root=DATA_ROOT)

    for model_name in MODELS:
        done += 1
        run_id = f'{model_name}_{feature_mode}'
        print(f'\n[{done}/{total}] Running {run_id} ...')
        result = train_run(model_name, feature_mode, run_id, train_loader, val_loader, normalizer)
        results.append(result)
        pd.DataFrame(results).to_csv(results_path, index=False)
        print(f'  Saved → {results_path}')

df = pd.DataFrame(results)
print('\n── Study 2 Results ──')
print(df[['run_id', 'model', 'feature_mode', 'best_val_mae', 'params']].to_string(index=False))

In [ ]:
# ── Cell 7: CVPR-style grouped bar chart ─────────────────────────────────────
import matplotlib, matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np, pandas as pd

matplotlib.rcParams.update({
    'font.family': 'serif', 'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 9, 'axes.titlesize': 9, 'axes.labelsize': 9,
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'figure.dpi': 300, 'axes.spines.top': False, 'axes.spines.right': False,
})

MODE_COLORS = {'topology': '#648FFF', 'full': '#FE6100'}
LABELS_MAP  = {'gcn': 'GCN', 'gat': 'GAT', 'gatv2': 'GATv2'}

df = pd.read_csv(f'{DRIVE_OUT}/results/study2_results.csv')

x      = np.arange(len(MODELS))
width  = 0.35
fig, ax = plt.subplots(figsize=(3.5, 2.6))

for i, mode in enumerate(FEATURE_MODES):
    vals = [df[(df['model'] == m) & (df['feature_mode'] == mode)]['best_val_mae'].values[0]
            for m in MODELS]
    offset = (i - 0.5) * width
    bars = ax.bar(x + offset, vals, width, label=mode.capitalize(),
                  color=MODE_COLORS[mode], edgecolor='white', linewidth=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                f'{v:.3f}', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels([LABELS_MAP[m] for m in MODELS])
ax.set_ylabel('Val MAE (Ha)')
ax.set_title('Topology vs. Full Node Features')
ax.legend(frameon=False)
fig.tight_layout(pad=0.4)

plot_path = f'{DRIVE_OUT}/plots/study2_feature_mode.pdf'
fig.savefig(plot_path, format='pdf', bbox_inches='tight')
fig.savefig(plot_path.replace('.pdf', '.png'), format='png', bbox_inches='tight', dpi=300)
plt.show()
print(f'Plot saved → {plot_path}')